In [ ]:
input_data = None
targetpop_data = None
output_data = None
output_model = None
util = None
display_util = None
configfile = "config/config.yml"

In [ ]:
import yaml

with open(configfile) as stream:
    config = yaml.safe_load(stream)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline
import matplotlib.pyplot as plt

from IPython.display import Markdown
import pandera.pandas as pa
from pandera.typing import Series

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)
plt.ioff()
plt.rcParams["figure.figsize"] = (8, 5)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import (  # noqa: E402
    rule_setup,
    display_data_doc,
    display_long_data_doc,
)
from util import (  # noqa: E402
    EmpfaengerID,
    common_translate,
    split_data,
    collapse_col,
    find_redundant_cols,
    fix_redundancies,
)

### Target Population Filtering

The recipients in the dataset were filtered to match the target population. We tried afterwards to remove empty and duplicate columns.

In [ ]:
data = pd.read_parquet(input_data)
rec = collapse_col(
    data.loc[:, ["recipient_et_iqtig", "recipient_et_id_et"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
targetpop = pd.read_parquet(targetpop_data)
data = data[rec.isin(targetpop["recipient_et_id_et"])]
display(
    Markdown(
        f"""The filter process reduced the number of recipients in the data ({rec.nunique()}) and target population ({targetpop["recipient_et_id_et"].nunique()})
            to {rec[rec.isin(targetpop["recipient_et_id_et"])].nunique()} in the processed data.
        """
    )
)

### Integration of Seperated Institute Data

In this file the {term}`IQTIG` and {term}`ET` not connected (see [](general:ic)). Rows from {term}`IQTIG` contain information on the dialysis of the patient. 

In [ ]:
idcols = ["recipient_et_iqtig", "recipient_et_id_et"]
assert len(split_data(data, idcols)) == 2, "Not 2 different row types present!?"

## Domain Steps

For this file the plan for domain preprocessing of longitudinal data was followed (see [](general:ds)).

### Row Filtering

We kept all rows (see [](general:rf)). For the following analysis we treated the `dialysis_start_date_iqtig` column as the date column together with `date` from {term}`ET` `date_combined` and combined the recipient ID columns.

In [ ]:
data["Institute"] = (
    (~data["recipient_et_id_et"].isna()) + (~data["recipient_et_iqtig"].isna()) * 2
).replace({1: "ET", 2: "IQTIG", 3: "ET+IQTIG", 0: "Neither ID"})
data["date_combined"] = collapse_col(
    data.loc[:, ["date", "dialysis_start_date_iqtig"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
data["recipient_id"] = collapse_col(
    data.loc[:, ["recipient_et_iqtig", "recipient_et_id_et"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
display_long_data_doc(
    data,
    [
        "recipient_id",
    ],
    "date_combined",
    "Institute",
)
data.drop(columns=["Institute", "date_combined", "recipient_id"], inplace=True)

### Unit Conversions

We applied common translations and removed unit specifier columns (see [](general:uc)).

In [ ]:
data = common_translate(data, config["data"]["common_translations"])

In [ ]:
cols = data.columns[data.columns.to_series().str.contains("_unit")]
assert (data[cols].nunique() != 1).sum() == 0
dropme = cols[data[cols].nunique() == 1]
data = data.drop(columns=dropme)
dropme = ", ".join((f"`{col}`" for col in dropme))
display(Markdown(f"The columns {dropme} were removed as only a single unit was used."))

### Consolidating Columns

We consolidated columns that appear for both {term}`ET` and {term}`IQTIG` (see [](general:crc)).

In [ ]:
red = find_redundant_cols(data)
red["recipient_et_id_et"] = ["recipient_et_iqtig", "recipient_et_id_et"]
fix_redundancies(data, red)

## Intermediate Dataset

For this longitudinal dataset we recommend the `date` and the `dialysis_start_date` columns as the time axis.

In [ ]:
indcols = ["recipient_et_id_et"]
data = data.sort_index(axis=1).sort_values(
    indcols + ["date", "dialysis_start_date"], axis=0
)
data = data.set_index(indcols)

In [ ]:
# Another base class might be necessary, see util.py
# describe columns, without checks for now, order is important
class WaitingListKidney(EmpfaengerID):
    acceptable_mismatch_program: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Acceptable Mismatch program",
        description="Was the patient in a acceptable mismatch program?",
        isin=["yes", "no"],
    )
    admission_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Admission Date",
        description="When was the patient admissioned to the center?",
    )
    date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Date",
        description="When was this data entry collected?",
    )
    dialysis: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Dialysis",
        description="Was the patient receiving dialysis?",
        isin=["yes", "no"],
    )
    dialysis_start_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Dialysis Start Date",
        description="When was dialysis started?",
    )
    dialysis_type: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Dialysis Type",
        description="What dialysis type was the patient receiving?",
        isin=[
            "Haemodialysis",
            "Peritoneal Dialysis (CAPD, APD, CCPD)",
            "Intermittent Peritoneal Dialysis",
            "Recipient not on dialysis",
            "Home Haemodialysis",
            "Recipient on dialysis, technique unknown",
        ],
    )
    in_maturation: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="In Maturation",
        description="?",
        isin=["yes", "no"],
    )
    in_maturation_detection_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="In Maturation Detection date",
        description="?",
    )
    kidney_base_disease_icd: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Kidney Disease ICD Code",
        description="Which was the ICD code of the diagnosed kidney disease?",
    )
    kidney_base_disease_icd: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Kidney Disease ICD Text",
        description="Which was the ICD text of the diagnosed kidney disease?",
    )
    kidney_base_disease_number: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Kidney Disease Number",
        description="?",
    )
    kidney_base_disease_icd: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Kidney Disease ICD Text",
        description="Which was the ICD text of the diagnosed kidney disease?",
    )
    kidney_base_disease_text: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Kidney Disease Text",
        description="What is the general description of the kidney disease?",
    )
    living_donor: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Living donor",
        description="Was this patient a living donor?",
        isin=["yes", "no"],
    )
    number_of_any_transplants: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Number of transplants",
        description="How many transplants did this patient receive?",
    )
    poss_donor_domino: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Domino Possible",
        description="Can this patient receive a domino transplant?",
        isin=["yes", "no"],
    )
    poss_donor_domino: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Enbloc Possible",
        description="Can this patient receive an enbloc transplant?",
        isin=["yes", "no"],
    )
    poss_donor_domino: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="ESP Possible",
        description="Can this patient receive an ESP transplant?",
        isin=["yes", "no"],
    )
    poss_donor_domino: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Extended Donor Pool Possible",
        description="Can this patient receive an extended donor pool transplant?",
        isin=["yes", "no"],
    )
    poss_donor_hepatitis_b_core_antibodies: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hepatitis B Core Donor Possible",
        description="Can this patient receive a transplant from a donor with hepatitis B core antibodies?",
        isin=["negative", "Any"],
    )
    poss_donor_hepatitis_b_surface_antigens: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hepatitis B Surface Donor Possible",
        description="Can this patient receive a transplant from a donor with hepatitis B surface antigens?",
        isin=["negative", "Any"],
    )
    poss_donor_hepatitis_c_antibodies: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hepatitis C Donor Possible",
        description="Can this patient receive a transplant from a donor with hepatitis C antibodies?",
        isin=["negative", "Any"],
    )
    poss_donor_malign_tumor: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Malign tumor possible",
        description="Can this patient receive a transplant from a donor with a malign tumor?",
        isin=["yes", "no"],
    )
    poss_donor_max_age: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Maximum donor age",
        description="How old can a possible donor be?",
    )
    poss_donor_mengitis: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Mengitis possible",
        description="Can this patient receive a transplant from a donor with a mengitis?",
        isin=["yes", "no"],
    )
    poss_donor_min_age: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Minimum donor age",
        description="How old should a possible donor be?",
    )
    poss_donor_sepsis: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Sepsis possible",
        description="Can this patient receive a transplant from a donor with a sepsis?",
        isin=["yes", "no"],
    )
    poss_donor_substance_abuse: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Substance abuse possible",
        description="Can this patient receive a transplant from a donor with a history of substance abuse?",
        isin=["yes", "no"],
    )
    pre_existing_disease: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Pre existing Disease",
        description="Had this patient a pre existing disease?",
        isin=["yes", "no"],
    )
    program: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Program",
        description="In which program was this patient?",
        isin=["ESP", "ETKAS"],
    )
    rejection_no_capacity: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Rejection due to no capacity",
        description="Was this recipient rejected due to no capacity?",
        isin=["yes", "no"],
    )
    rejection_no_capacity: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Rejection due to no capacity",
        description="Was this recipient rejected due to no capacity?",
        isin=["yes", "no"],
    )
    waiting_state: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Waiting State",
        description="Waiting state of the patient on this list",
        isin=["FU_KI", "NT_KI", "R_KI", "D_KI", "T_KI", "I_KI", "HI_KI"],
    )
    waiting_state_full: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Full Waiting State",
        description="Full Waiting state of the patient on this list",
        isin=[
            "FU - Transplanted",
            "NT - Not Transplantable",
            "R - Removed from Waiting List",
            "D - Deceased",
            "T - Transplantable (allo-PRA% 0 - 5)",
            "I - Immunized (allo-PRA% 6 - 84)",
            "HI - Highly Immunized (allo-PRA% 85 - 100)",
        ],
    )
    waiting_state_reason: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Reason for the waiting state",
        description="Why is this patient in this state?",
    )

    class Config:
        title = "Kidney Waitlist Dataset"
        description = "Each row represents a data update relevant for the kidney waiting list. The data is based on the 'element_warteliste_niere.csv' file. It contains data from the ET and IQTIG."
        multiindex_strict = True
        multiindex_coerce = True

In [ ]:
display_data_doc(WaitingListKidney, data)

In [ ]:
WaitingListKidney.to_schema().validate(data).to_parquet(output_data)
with open(output_model, "wt") as fh:
    WaitingListKidney.to_yaml(stream=fh)

## Technical Information

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "output_model": output_model,
        "util": util,
        "display_util": display_util,
    }
)